In [1]:
from pathlib import Path
from langchain_ollama import ChatOllama, OllamaEmbeddings

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print(len(embeddings.embed_query("test")), "dimensions")

768 dimensions


In [4]:
dossier = ROOT / "data" / "pdfs"
print("Dossier :", dossier)
print("Existe :", dossier.exists())
print("Contenu :", list(dossier.iterdir()) if dossier.exists() else "—")

Dossier : c:\Users\aliel\Desktop\tp5_agentic\agentic-ai-mcp-tp5\data\pdfs
Existe : True
Contenu : []


In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyPDFDirectoryLoader(str(ROOT / "data" / "pdfs")).load()
print(len(pages), "pages chargées")

decoupeur = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = decoupeur.split_documents(pages)
print(len(chunks), "chunks")
print(chunks[0].metadata["source"], "| page", chunks[0].metadata["page"])

15 pages chargées
52 chunks
c:\Users\aliel\Desktop\tp5_agentic\agentic-ai-mcp-tp5\data\pdfs\attention_is_all_you_need.pdf | page 0


In [6]:
from langchain_chroma import Chroma

vectorstore = Chroma(
    collection_name="documents",
    embedding_function=embeddings,
    persist_directory=str(ROOT / "chroma_db"),   # sauvegardée sur le disque (ignorée par Git)
)

if vectorstore._collection.count() == 0:          # on n'indexe qu'une seule fois
    vectorstore.add_documents(chunks)

print(vectorstore._collection.count(), "chunks indexés")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

52 chunks indexés


In [7]:
for doc in retriever.invoke("What is multi-head attention?"):
    print(f"[{Path(doc.metadata['source']).name}, page {doc.metadata['page'] + 1}]")
    print("  ", doc.page_content[:150].replace("\n", " "), "...\n")

[attention_is_all_you_need.pdf, page 5]
   output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows th ...

[attention_is_all_you_need.pdf, page 5]
   is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head att ...

[attention_is_all_you_need.pdf, page 2]
   in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more difficult to learn dependencies between di ...

[attention_is_all_you_need.pdf, page 4]
   Scaled Dot-Product Attention  Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several att ...



In [8]:
from langchain.tools import tool

@tool
def rechercher_documents(requete: str) -> str:
    """Recherche des passages dans les documents PDF indexés (notamment l'article
    "Attention Is All You Need" sur le Transformer).
    À utiliser pour toute question sur le contenu de ces documents.
    La requête doit être courte, précise et rédigée en anglais."""
    documents = retriever.invoke(requete)
    if not documents:
        return "Aucun passage trouvé."
    return "\n\n".join(
        f"[{Path(d.metadata['source']).name}, page {d.metadata['page'] + 1}] {d.page_content}"
        for d in documents
    )

print(rechercher_documents.invoke("positional encoding")[:500])

[attention_is_all_you_need.pdf, page 6] tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
learned and fixed [9].
In this work, we use sine and cosine functions of different frequencies:
P E(pos,2i) = sin(pos/100002i/dmodel )
P E(pos,2i+1) = cos(pos/100002


In [9]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

agent_rag = create_agent(
    model=llm,
    tools=[rechercher_documents],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Tu es un assistant documentaire. "
        "Pour toute question sur le contenu des documents, utilise l'outil rechercher_documents "
        "avant de répondre, et ne réponds pas de mémoire. "
        "Cite tes sources sous la forme [fichier, page X]. "
        "Si l'information n'est pas dans les documents, dis-le clairement. "
        "Pour une simple salutation, réponds directement sans outil. "
        "Réponds en français."
    ),
)

def discuter(question, thread_id="chat-1"):
    config = {"configurable": {"thread_id": thread_id}}
    resultat = agent_rag.invoke({"messages": [HumanMessage(question)]}, config=config)
    outils = [a["name"] for m in resultat["messages"][-4:] if getattr(m, "tool_calls", None) for a in m.tool_calls]
    print(f"\n👤 {question}")
    print(f"   🔧 outils appelés : {outils or 'aucun'}")
    print(f"🤖 {resultat['messages'][-1].content}")

In [10]:
discuter("Bonjour !")
discuter("Qu'est-ce que l'attention multi-têtes dans le Transformer ?")
discuter("Et combien de têtes utilisent-ils dans le modèle de base ?")   # « ils » = mémoire
discuter("Quel est le score MMLU de GPT-4 selon ce document ?")          # info absente


👤 Bonjour !
   🔧 outils appelés : aucun
🤖 Bonjour ! 😊 Comment puis-je vous aider aujourd'hui ?

👤 Qu'est-ce que l'attention multi-têtes dans le Transformer ?
   🔧 outils appelés : ['rechercher_documents']
🤖 L'attention multi-têtes dans le Transformer permet au modèle de traiter simultanément différentes représentations de l'entrée en se concentrant sur des sous-espaces de représentation différents. Selon l'article "Attention Is All You Need", le modèle utilise 8 têtes d'attention (heads) avec une dimension de 64 pour chaque tête. Cette architecture est appliquée de trois manières :  
1. L'attention encodeur-décodeur (les requêtes proviennent du décodeur précédent, les clés et valeurs du encodeur),  
2. Les couches d'attention auto-encodées dans l'encodeur,  
3. Les couches d'attention auto-décodées dans le décodeur.  
Cela permet de surmonter les limites des modèles précédents en réduisant le coût computationnel tout en conservant une résolution efficace. [attention_is_all_you_need.pd

In [11]:
print("Chatbot RAG — tape 'quit' pour arrêter")
while True:
    question = input("Vous : ")
    if question.strip().lower() in ("quit", "exit", ""):
        break
    discuter(question, thread_id="chat-interactif")

Chatbot RAG — tape 'quit' pour arrêter

👤 enter
   🔧 outils appelés : aucun
🤖 Bonjour ! Comment puis-je vous aider aujourd'hui ?

👤 est ce que tu comprends ma question 
   🔧 outils appelés : aucun
🤖 Oui, je comprends votre question. Comment puis-je vous aider aujourd'hui ?

👤 c'est bon quit 
   🔧 outils appelés : aucun
🤖 Au revoir ! 😊
